# ch01 图像读写与数组基础（练习版）

> 按提示补全 `____`，跑通所有 assert。`cam` / `home` / `mess` 已在 setup 里读好，
> `probe` / `n_diff` / `tmp_dir` 脚手架可用。
>
> 注意：挖空块内每条语句保持**单行**；循环 / 分支写在挖空块之外。
> 题目真值都来自实跑（opencv 5.0.0），照提示写就能对上。

In [ ]:
from pathlib import Path

import shutil
import warnings

import numpy as np

warnings.simplefilter("ignore")

import cv2
from PIL import Image

DATA = Path("data")
CL = DATA / "classic"

In [ ]:
# 三张代表图
cam = cv2.imread(str(CL / "camera.png"))    # 存成 PNG 的灰度图（读进来仍是 3 通道）
home = cv2.imread(str(CL / "home.jpg"))     # 真彩色 JPEG（B-R 平均差 92.7，通道差异明显）
mess = cv2.imread(str(CL / "messi5.jpg"))   # JPEG 往返素材
print("cv2", cv2.__version__, "| camera", cam.shape, "| home", home.shape, "| messi5", mess.shape)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def n_diff(a, b):
    """统计两个同形状数组中有多少个元素不相等（逐通道计数）。"""
    return int((np.asarray(a) != np.asarray(b)).sum())


tmp_dir = Path("_tmp_io")     # 落盘往返用，收尾会删掉
if not tmp_dir.exists():
    tmp_dir.mkdir()
print("脚手架就绪：probe / n_diff / tmp_dir")

## 任务 1：元信息与灰度读取

`shape` / `dtype` / `nbytes` + `IMREAD_GRAYSCALE`。

In [ ]:
# TODO(1)：元信息：shape / dtype / nbytes，再用灰度模式读同一张图
# 提示：cv2.imread(路径) 默认 IMREAD_COLOR → (H, W, 3)；cv2.IMREAD_GRAYSCALE → (H, W)
cam_shape = ____
cam_dtype = ____
cam_bytes = ____
gray_cam = ____
print("camera:", cam_shape, cam_dtype, cam_bytes, "bytes | 灰度读:", gray_cam.shape)
print("解读: nbytes = H*W*C —— 512*512*3 = 786432，这是内存占用的最小估算")

assert cam_shape == (512, 512, 3)
assert cam_dtype == "uint8" and cam_bytes == 786432
assert gray_cam.shape == (512, 512), "灰度读少一个维度：H x W"

## 任务 2：读失败怎么办

cv2 返回 `None` vs PIL 抛异常。

In [ ]:
# TODO(2)：读失败会怎样：cv2 静默返回 None，PIL 直接抛 FileNotFoundError
# 提示：cv2.imread 读不存在/非图片 → None；PIL 用脚手架 probe(Image.open, 路径) 接住异常
missing_cv = ____
not_image = ____
st_pil, err_pil = ____
print("cv2 读不存在的图:", missing_cv, "| cv2 读文本文件:", not_image)
print("PIL 读不存在的图 →", st_pil, err_pil)
print("解读: cv2 读失败不抛异常，只给 None。imread 之后必须判 None，"
      "否则后面 img.shape 报 AttributeError，报错位置离现场很远")

assert missing_cv is None and not_image is None
assert st_pil == "err" and err_pil == "FileNotFoundError"

## 任务 3：BGR 与 RGB 对账

首像素互为倒序，全图翻转一致。

In [ ]:
# TODO(3)：通道顺序对账：cv2 是 BGR，PIL 是 RGB，首像素必须互为倒序
# 提示：np.array(Image.open(路径)) 得到 RGB；home[..., ::-1] 把 BGR 翻成 RGB
pil_home = ____
pix_bgr = ____
pix_rgb = ____
flip_ok = ____
max_diff = ____
print("cv2 首像素 BGR:", pix_bgr, "| PIL 首像素 RGB:", pix_rgb)
print("全图翻转后与 PIL 完全一致:", flip_ok, "| 最大差:", max_diff)
print("解读: 这条是 CV 第一坑——用 cv2 读图接 PIL/matplotlib 显示，不翻通道就会"
      "『人脸变阿凡达』。记法：cv2 存的是 BGR，显示库要 RGB")

assert pix_bgr == [164, 102, 31] and pix_rgb == [31, 102, 164]
assert flip_ok and max_diff == 0

## 任务 4：通道拆合与统计

`split` / `merge` + 三通道均值。

In [ ]:
# TODO(4)：通道拆合：split 出 B/G/R 看均值（这张图偏蓝），merge 必须能还原
# 提示：b, g, r = cv2.split(img)；cv2.merge([b, g, r])
b_ch, g_ch, r_ch = ____
b_mean = ____
g_mean = ____
r_mean = ____
merge_ok = ____
print("通道均值 B/G/R:", b_mean, g_mean, r_mean, "| merge(split(x)) == x:", merge_ok)
print("解读: 三通道均值不相等才说明这是真彩图；camera.png 三通道完全相同"
      "（灰度存成 PNG），拿它做通道分析会看不出任何差异")

assert (b_mean, g_mean, r_mean) == (141.2399, 123.8721, 91.6084)
assert merge_ok

## 任务 5：灰度化两把尺子

`cvtColor` vs 手写加权 vs 三通道平均。

In [ ]:
# TODO(5)：灰度化两把尺子：cvtColor 加权公式 vs 三通道平均
# 提示：cvtColor(img, cv2.COLOR_BGR2GRAY)；官方权重 0.299R + 0.587G + 0.114B（R 在前！）
gray_cv = ____
gray_weighted = ____
gray_avg = ____
d_weighted = ____
d_avg_max = ____
d_avg_mean = ____
n_trunc_vs_cv = ____
n_trunc_vs_round = ____
gray_mean = ____
print("手写加权 vs cvtColor 最大差:", d_weighted)
print("三通道平均 vs cvtColor: 最大差", d_avg_max, "| 平均差", d_avg_mean)
print("平均法直接 astype 截断后与 cvtColor 不同像素数:", n_trunc_vs_cv, "/", gray_cv.size)
print("解读: 加权公式等价于 cvtColor（差 0）；三通道『平均』是另一把尺子，"
      "最大差 20.33 —— 人眼对绿色最敏感，这就是权重 0.587 的来历")

assert d_weighted == 0, "手写加权公式与 cvtColor 完全等价"
assert (d_avg_max, d_avg_mean) == (20.3333, 9.8214)
assert n_trunc_vs_cv == 195079 and n_trunc_vs_round == 65393, "float→uint8 是截断不是四舍五入"
assert gray_mean == 116.195

## 任务 6：归一化与回写

float32 归一化、截断与四舍五入。

In [ ]:
# TODO(6)：归一化到 [0,1] 再回写 uint8：会不会丢精度？
# 提示：img.astype(np.float32) / 255.0；回写 (f * 255).astype(np.uint8)
f32 = ____
back_trunc = ____
back_round = ____
n_back_trunc = ____
n_back_round = ____
f_min = ____
f_max = ____
print(f32.dtype, "范围", f_min, "~", f_max, "| 占用", f32.nbytes, "bytes（uint8 的 4 倍）")
print("回写差异：截断", n_back_trunc, "| 四舍五入", n_back_round)
print("解读: n/255 再 *255 在 float32 下可逆，回写零损失；但换成『先算别的运算再回写』"
      "（如 gamma、均值），截断就会系统性压暗 —— 记住原则：回写前一律 round/clip")

assert (f_min, f_max) == (0.0, 1.0)
assert f32.nbytes == 2359296
assert n_back_trunc == 0 and n_back_round == 0, "纯归一化往返零损失"

## 任务 7：resize 三插值

AREA / LINEAR / NEAREST + fx/fy。

In [ ]:
# TODO(7)：resize 三插值：缩到 128 时 AREA / LINEAR / NEAREST 的差距
# 提示：cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA)；fx/fy 走比例缩放
small_area = ____
small_linear = ____
small_nearest = ____
d_linear_area = ____
d_nearest_area = ____
shape_half = ____
shape_big = ____
print("缩小 128 平均差：LINEAR-AREA", d_linear_area, "| NEAREST-AREA", d_nearest_area)
print("fx=0.5 →", shape_half, "| fx=1.5 (CUBIC) →", shape_big)
print("解读: 缩小用 INTER_AREA（像素面积平均，抗混叠），放大用 LINEAR/CUBIC；"
      "NEAREST 在缩小时直接丢像素（平均差 8.86，出现锯齿）")

assert small_area.shape == (128, 128, 3)
assert (d_linear_area, d_nearest_area) == (4.2628, 8.8559)
assert shape_half == (192, 256, 3) and shape_big == (576, 768, 3)

## 任务 8：uint8 溢出

回绕 vs 饱和，精确对账。

In [ ]:
# TODO(8)：像素加减的溢出：numpy 回绕 vs cv2 饱和，哪个是你想要的
# 提示：(img + 100).astype(np.uint8) 会回绕变暗；cv2.add(img, 100) 饱和在 255
wrap_add = ____
sat_add = ____
n_wrap_dark = ____
n_sat_dark = ____
n_sat_255 = ____
n_gt155 = ____
wrap_sub = ____
n_sub_bright = ____
n_sub_0 = ____
n_lt100 = ____
print("+100：回绕变暗的通道值", n_wrap_dark, "| 饱和写法变暗", n_sat_dark,
      "| 饱和到 255 的通道值", n_sat_255)
print("-100：回绕变亮的通道值", n_sub_bright, "| 饱和到 0 的通道值", n_sub_0)
print("解读: 回绕数 == 超过阈值（>155）的通道值数 =", n_gt155,
      "——uint8 加减法是模 256 运算，262 会变成 6。调亮度一律用 cv2.add/subtract")

assert n_wrap_dark == n_gt155 == 212145, "回绕数恰好等于越界像素数"
assert n_sat_dark == 0 and n_sat_255 == 215325
assert n_sub_bright == n_lt100 == 224627 and n_sub_0 == 228327

## 任务 9：gamma 校正

非线性提亮与不动点。

In [ ]:
# TODO(9)：gamma 校正：把暗图提亮的非线性做法
# 提示：((img / 255.0) ** gamma) —— gamma < 1 提亮，> 1 压暗；算完 clip 回 [0,255]
gamma_img = ____
g_mean_before = ____
g_mean_after = ____
n_brightened = ____
n_darkened = ____
print("均值", g_mean_before, "→", g_mean_after, "| 提亮通道值", n_brightened, "| 压暗", n_darkened)
print("解读: gamma=0.5 让整体变亮；0 和 255 是幂运算的不动点，所以纯黑纯白不变。"
      "『均值直方图都对了』是调亮的标准验收")

assert (g_mean_before, g_mean_after) == (118.9068, 165.7185)
assert n_darkened == 0, "gamma<1 不会压暗任何像素"

## 任务 10：ROI、掩码与直方图

视图语义 + 掩码统计 + 自检公式。

In [ ]:
# TODO(10)：ROI 切片与掩码统计：只关心感兴趣区域
# 提示：切片 img[y1:y2, x1:x2] 是视图（改 ROI 会改原图）；cv2.mean(img, mask) 带掩码统计
roi = ____
roi_mean = ____
full_mean = ____
gray_home = ____
mask = ____
mask[100:200, 200:300] = ____
masked_mean = ____
hist = ____
print("ROI", roi.shape, "均值", roi_mean, "| 全图均值", full_mean, "| 掩码内灰度均值", masked_mean)
print("直方图 sum:", int(hist.sum()), "== 像素数", gray_home.size,
      "| 峰值 bin", int(hist.argmax()), "计数", int(hist.max()))
print("解读: ROI 是视图不是拷贝（roi[:] = 0 会连带改原图，要隔离用 .copy()）；"
      "直方图 256 个 bin 的计数总和恒等于像素数，这是自检公式")

assert roi.shape == (100, 100, 3)
assert roi_mean == 103.233 and full_mean == 118.9068
assert masked_mean == 93.2983
assert int(hist.sum()) == gray_home.size == 196608
assert int(hist.argmax()) == 111 and int(hist.max()) == 3908

## 任务 11：落盘往返

PNG 无损、JPEG q90/q50 有损。

In [ ]:
# TODO(11)：落盘往返：PNG 无损、JPEG 有损，q90 与 q50 各差多少
# 提示：cv2.imwrite(路径, img, [cv2.IMWRITE_JPEG_QUALITY, 90])；再 imread 回来对账
png_path = ____
jpg90_path = ____
jpg50_path = ____
____
____
____
back_png = ____
back_jpg90 = ____
back_jpg50 = ____
n_png = ____
n_jpg90 = ____
n_jpg50 = ____
max_jpg90 = ____
size_png = ____
size_jpg90 = ____
size_jpg50 = ____
print("体积 png", size_png, "| jpg q90", size_jpg90, "| jpg q50", size_jpg50)
print("不同通道值数 png", n_png, "| q90", n_jpg90, "| q50", n_jpg50, "（总", mess.size, "）")
print("解读: PNG 逐位无损（0 差异）；JPEG 有损且压缩比越高丢得越多——"
      "所以『标签/掩码图必须存 PNG』，而竞赛交付图片素材通常存 JPEG 省体积")

assert n_png == 0, "PNG 无损：往返 0 差异"
assert n_jpg90 == 424160 and n_jpg50 == 494055
assert max_jpg90 == 46
assert size_jpg90 == 53807 and size_png == 316930 and size_jpg50 == 19634

shutil.rmtree(tmp_dir, ignore_errors=True)   # 清理实验文件
print("已清理", tmp_dir)

## 自查清单

- [ ] 能说出 cv2 与 PIL 的通道顺序差异，并写出两种翻转写法
- [ ] 能解释 `imread` 返回 `None` 的后果与防御写法
- [ ] 能手写灰度加权公式（R/G/B 系数别记反）
- [ ] 知道 `resize` 的参数是 `(宽, 高)`
- [ ] 能解释 uint8 溢出为什么等于「越界像素数」
- [ ] 知道掩码图必须存 PNG